# 03 — Summary (OAT + Group study 종합)

01_oat 와 02_group_study 결과를 종합해 발표 자료 4종 생성. **둘 다 끝난 후 1회 실행**.

- **입력**: `4_output/baseline/oat/master.csv`, `4_output/baseline/group/{optuna.db, param_importance.csv}`
- **출력**: `4_output/baseline/summary/{tornado.png, group_importance.png, comparison.csv, summary_report.png}`
- **참조**: [strategy.md §8·§9](strategy.md)

## 1. 환경 + 산출물 로드

In [ ]:
import os, sys

# Colab이면 setup만 실행, 로컬이면 ../../setup.py
try:
    import google.colab
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
    PROJECT_ROOT = '/content/project'
except ImportError:
    %run ../../setup.py
    from utils.config import PROJECT_ROOT

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna

# 01_oat / 02_group_study가 저장한 산출물 경로
BASE = os.path.join(PROJECT_ROOT, '4_output', 'baseline')
OAT_CSV       = os.path.join(BASE, 'oat',   'master.csv')          # OAT 결과 행들
GROUP_DB      = os.path.join(BASE, 'group', 'optuna.db')           # group study DB
GROUP_IMP_CSV = os.path.join(BASE, 'group', 'param_importance.csv')# fANOVA 축 중요도
SUMMARY_DIR   = os.path.join(BASE, 'summary')                      # 발표 산출물 출력 폴더
os.makedirs(SUMMARY_DIR, exist_ok=True)

oat = pd.read_csv(OAT_CSV)
group_imp = pd.read_csv(GROUP_IMP_CSV)
study = optuna.load_study(study_name='baseline_group', storage=f'sqlite:///{GROUP_DB}')

print(f'OAT master: {len(oat)} rows, axes={oat["axis"].nunique()}')
print(f'Group study: {len(study.trials)} trials')
print(f'Group importance: {len(group_imp)} axes')

## 2. OAT — Tornado plot

축별 RMSE range (max - min) 정렬, 가로 bar. seed 평균으로 안정화.

In [ ]:
# 각 (axis, option)의 val_rmse를 seed 5개 평균으로 안정화
ablation = (
    oat[oat['axis'] != 'reference']
    .groupby(['axis', 'option'])['val_rmse']
    .mean()
    .reset_index()
)
ref_val = oat[oat['axis'] == 'reference']['val_rmse'].mean()   # 기준점 RMSE (reference cfg)

# 축별 영향력 = 그 축 옵션들 사이의 val_rmse range(max - min). 작은 순으로 정렬
axis_range = (
    ablation.groupby('axis')['val_rmse']
    .agg(lambda s: s.max() - s.min())
    .sort_values()
)

# 가로 막대(tornado): 위로 갈수록 영향력 큰 축
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(axis_range.index, axis_range.values, color='steelblue')
ax.set_xlabel('RMSE range (max − min within axis)')
ax.set_title(f'OAT Tornado — axis별 marginal 영향력 (reference RMSE = {ref_val:.6f})')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(os.path.join(SUMMARY_DIR, 'tornado.png'), dpi=120, bbox_inches='tight')
plt.show()

print('axis별 marginal Δ:')
for ax_name, delta in axis_range[::-1].items():   # 영향력 큰 축부터 출력
    print(f'  {ax_name:20s} Δ {delta:.6f}')

## 3. Group study — Param importance

In [ ]:
# group study의 fANOVA 중요도를 가로 막대로 (작은 순 정렬 → 위로 갈수록 중요)
imp_sorted = group_imp.sort_values('importance')

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(imp_sorted['axis'], imp_sorted['importance'], color='coral')
ax.set_xlabel('Param importance (fANOVA)')
ax.set_title(f'Group study Param Importance ({len(study.trials)} trials, multivariate TPE)')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(os.path.join(SUMMARY_DIR, 'group_importance.png'), dpi=120, bbox_inches='tight')
plt.show()

## 4. 비교표 (comparison.csv)

OAT marginal best vs Group study best. 일치하면 marginal 신뢰성 ↑.

In [ ]:
# OAT 쪽 "축별 최적 옵션": 그 축에서 val_rmse가 가장 낮은 option과 그 값
oat_min = ablation.sort_values('val_rmse').groupby('axis').first()
oat_best_option = oat_min['option'].rename('oat_marginal_best')
oat_best_rmse   = oat_min['val_rmse'].rename('oat_best_val_rmse')
ref_val_series  = pd.Series({ax: ref_val for ax in oat_best_option.index},
                            name='reference_val_rmse')

# Group study 쪽 "최적": best trial이 고른 축별 값 + 그 trial의 oof/val/test RMSE
best_trial = study.best_trial
group_best_option = pd.Series(best_trial.params, name='group_best')
group_best_oof = pd.Series({ax: best_trial.value for ax in group_best_option.index},
                           name='group_best_oof_rmse')
group_best_val = pd.Series(
    {ax: best_trial.user_attrs.get('val_rmse') for ax in group_best_option.index},
    name='group_best_val_rmse',
)
group_best_test = pd.Series(
    {ax: best_trial.user_attrs.get('test_rmse') for ax in group_best_option.index},
    name='group_best_test_rmse',
)

# 두 방식의 축별 선택을 나란히 놓고 일치 여부(agree) 표시 → OAT marginal의 신뢰성 점검
comparison = pd.concat(
    [oat_best_option, oat_best_rmse, ref_val_series,
     group_best_option, group_best_oof, group_best_val, group_best_test],
    axis=1,
)
comparison['agree'] = (comparison['oat_marginal_best'].astype(str)
                       == comparison['group_best'].astype(str))
comparison.to_csv(os.path.join(SUMMARY_DIR, 'comparison.csv'))
comparison

## 5. 발표용 종합 1장 (summary_report.png)

tornado + param importance + 핵심 메트릭을 1장에.

In [ ]:
fig, axes_pl = plt.subplots(1, 2, figsize=(16, 7))

# 좌: OAT tornado (위 cell 4와 동일 데이터)
axes_pl[0].barh(axis_range.index, axis_range.values, color='steelblue')
axes_pl[0].set_xlabel('RMSE range')
axes_pl[0].set_title(f'OAT marginal\n(reference RMSE = {ref_val:.6f})')
axes_pl[0].grid(alpha=0.3, axis='x')

# 우: Group study fANOVA 중요도 (위 cell 6과 동일 데이터)
axes_pl[1].barh(imp_sorted['axis'], imp_sorted['importance'], color='coral')
axes_pl[1].set_xlabel('Param importance (fANOVA)')
axes_pl[1].set_title(f'Group study\n({len(study.trials)} trials, best RMSE = {study.best_value:.6f})')
axes_pl[1].grid(alpha=0.3, axis='x')

fig.suptitle('Baseline Ablation — OAT marginal vs Group study importance',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SUMMARY_DIR, 'summary_report.png'), dpi=120, bbox_inches='tight')
plt.show()

print(f'\n발표 산출물 4종 → {SUMMARY_DIR}')
for f in ['tornado.png', 'group_importance.png', 'comparison.csv', 'summary_report.png']:
    p = os.path.join(SUMMARY_DIR, f)
    print(f'  {f:30s} {os.path.getsize(p)/1024:.1f} KB')